# L27 — Replications, Confidence Intervals, and Sample Size Planning

**Module**: M08 | **Chapter**: 10 | **Lecture**: L27

## Learning Objectives
By the end of this notebook you will be able to:
1. Compute a valid 95% confidence interval for a simulation output using independent replications.
2. Apply the sequential sample-size formula to achieve a target half-width.
3. Distinguish precision (CI width) from accuracy (bias) in simulation output.
4. Use `np.random.default_rng.spawn` to generate independent streams for replications.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

Continuing with M/M/1 (λ=0.8, μ=1.0, Wq*=4.0 min).
Warmup = 500 customers (from L26).
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import simpy
from scipy import stats

In [ ]:
LAM, MU    = 0.8, 1.0
WQ_STAR    = LAM / (MU * (MU - LAM))   # 4.0
WARMUP     = 500    # customers to delete
N_CUST     = 5_000  # customers per replication

def single_rep(lam, mu, n_customers, warmup, rng):
    """One replication: return Wq (excluding warmup)."""
    env    = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    waits  = []

    def customer():
        t0 = env.now
        with server.request() as req:
            yield req
            waits.append(env.now - t0)
            yield env.timeout(rng.exponential(1.0 / mu))

    def arrivals():
        for _ in range(n_customers):
            env.process(customer())
            yield env.timeout(rng.exponential(1.0 / lam))

    env.process(arrivals())
    env.run()
    return np.array(waits[warmup:]).mean()


print(f"Single replication Wq (seed=42): ", end="")
rng0 = np.random.default_rng(42)
print(f"{single_rep(LAM, MU, N_CUST, WARMUP, rng0):.3f} min")

## 1. Independent Replications Using `spawn`

Each replication must use statistically independent random streams.
`rng.spawn(n)` creates n child generators that are guaranteed to produce
non-overlapping sequences from the same seed.

In [ ]:
def run_replications(lam, mu, n_customers, warmup, n_reps, master_seed=0):
    """Run n_reps independent replications, return array of Wq estimates."""
    master_rng = np.random.default_rng(master_seed)
    child_rngs = master_rng.spawn(n_reps)
    return np.array([
        single_rep(lam, mu, n_customers, warmup, child_rngs[i])
        for i in range(n_reps)
    ])


# Pilot study: n0 = 10 replications
N0 = 10
pilot = run_replications(LAM, MU, N_CUST, WARMUP, n_reps=N0, master_seed=0)

m0 = pilot.mean()
s0 = pilot.std(ddof=1)
t_crit = stats.t.ppf(0.975, df=N0-1)
h0 = t_crit * s0 / np.sqrt(N0)

print(f"Pilot (n={N0}):")
print(f"  Wq̄  = {m0:.3f} min")
print(f"  s   = {s0:.3f}")
print(f"  95% CI: [{m0-h0:.3f}, {m0+h0:.3f}]  half-width h = {h0:.3f}")
print(f"  True Wq* = {WQ_STAR:.3f}  {'✓ covered' if m0-h0<=WQ_STAR<=m0+h0 else '✗ not covered'}")

## 2. Sequential Sample-Size Formula

We want 95% CI half-width $h^* \leq 0.25$ min.

$$n^* = \left\lceil \left( \frac{t_{n_0-1,\,0.025} \cdot s}{h^*} \right)^2 \right\rceil$$

In [ ]:
H_STAR = 0.25   # target half-width

n_star = int(np.ceil((t_crit * s0 / H_STAR) ** 2))
print(f"Required replications for h*={H_STAR}: n* = {n_star}")

# Cap at 200 for computational tractability in class
n_run = min(n_star, 200)
print(f"Running {n_run} replications (capped at 200)...")

results = run_replications(LAM, MU, N_CUST, WARMUP, n_reps=n_run, master_seed=1)

m_final = results.mean()
s_final = results.std(ddof=1)
t_final = stats.t.ppf(0.975, df=n_run-1)
h_final = t_final * s_final / np.sqrt(n_run)

print(f"\nFinal ({n_run} reps):")
print(f"  Wq̄  = {m_final:.3f}")
print(f"  95% CI: [{m_final-h_final:.3f}, {m_final+h_final:.3f}]  h = {h_final:.3f}")
print(f"  Target h* = {H_STAR}  {'✓ achieved' if h_final <= H_STAR else '✗ not achieved'}")
print(f"  True Wq* = {WQ_STAR:.3f}  {'✓ covered' if m_final-h_final<=WQ_STAR<=m_final+h_final else '✗ not covered'}")

## 3. CI Coverage Experiment

If the t-interval is valid, it should contain the true Wq* in 95% of experiments.

In [ ]:
N_EXPERIMENTS = 200   # independent experiments
N_PER_EXP     = 20   # replications per experiment

coverages = []
ci_los, ci_his, means_exp = [], [], []

for exp in range(N_EXPERIMENTS):
    reps = run_replications(LAM, MU, N_CUST, WARMUP, n_reps=N_PER_EXP,
                            master_seed=1000 + exp)
    m = reps.mean()
    h = stats.t.ppf(0.975, df=N_PER_EXP-1) * reps.std(ddof=1) / np.sqrt(N_PER_EXP)
    covered = (m - h) <= WQ_STAR <= (m + h)
    coverages.append(covered)
    ci_los.append(m - h); ci_his.append(m + h); means_exp.append(m)

coverage_rate = np.mean(coverages)
print(f"Coverage rate: {coverage_rate:.3f}  (nominal = 0.95)")

# Plot first 50 CIs
fig, ax = plt.subplots(figsize=(10, 6))
plot_n = 50
for i in range(plot_n):
    col = 'steelblue' if coverages[i] else 'tomato'
    ax.plot([ci_los[i], ci_his[i]], [i, i], color=col, lw=1.5)
    ax.plot(means_exp[i], i, 'o', color=col, ms=3)

ax.axvline(WQ_STAR, color='black', lw=1.5, linestyle='--', label=f'Wq*={WQ_STAR}')
ax.set_xlabel('Wq (min)')
ax.set_ylabel('Experiment index')
ax.set_title(f'95% CI coverage — first {plot_n} experiments  (rate={coverage_rate:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Single Long Run vs. Independent Replications

A single run with 100,000 customers can give a very narrow CI
using batch means — but there are important differences.

In [ ]:
# Approach 1: 30 independent replications of 5000 customers
reps_30 = run_replications(LAM, MU, 5_000, WARMUP, n_reps=30, master_seed=42)
m1 = reps_30.mean()
h1 = stats.t.ppf(0.975, 29) * reps_30.std(ddof=1) / np.sqrt(30)

# Approach 2: 1 run of 150,000 customers, batch means
rng_single = np.random.default_rng(42)
long_waits = []
env_s = simpy.Environment()
srv_s = simpy.Resource(env_s, capacity=1)

def cust_s():
    t0 = env_s.now
    with srv_s.request() as req:
        yield req
        long_waits.append(env_s.now - t0)
        yield env_s.timeout(rng_single.exponential(1.0/MU))

def arr_s():
    for _ in range(150_000):
        env_s.process(cust_s())
        yield env_s.timeout(rng_single.exponential(1.0/LAM))

env_s.process(arr_s())
env_s.run()

# batch means from long run (delete warmup)
lw = np.array(long_waits[WARMUP:])
bs = 100
nb = len(lw) // bs
bm = lw[:nb*bs].reshape(nb, bs).mean(axis=1)
m2 = bm.mean()
h2 = stats.t.ppf(0.975, nb-1) * bm.std(ddof=1) / np.sqrt(nb)

print("Approach                        | Mean Wq | Half-width | Total customers")
print("-" * 65)
print(f"30 independent reps × 5000      | {m1:.3f}  | {h1:.4f}     | {30*5000:,}")
print(f"1 long run × 150k (batch means) | {m2:.3f}  | {h2:.4f}     | {150_000:,}")
print()
print(f"True Wq* = {WQ_STAR:.3f}")
print("Note: batch means CI is narrower but assumes batches are uncorrelated.")

---
## Try It Yourself

1. **Higher load**: Repeat the coverage experiment at ρ=0.95 (λ=0.95). Does the 95% CI still achieve nominal coverage? Why might the warmup period need to be longer?

2. **Half-width targeting**: Suppose you need h*=0.10 min (more precise than 0.25). Compute n* from the pilot. Is it practical to run that many replications? What else could you do to reduce CI width without more replications?

3. **Batch autocorrelation**: Plot the autocorrelation function (ACF) of the batch means from Approach 2 above. Do consecutive batches appear to be correlated? What batch size is needed to reduce autocorrelation below 0.1? (Use `numpy.correlate` or `pandas.Series.autocorr`.)